# SUPARCO Soil Composition Model — Training Notebook
**Run on Google Colab with GPU (T4/A100)**

This notebook trains a ResNet-18 regression model to predict heavy metal concentrations
(Cd, Cu, Ni, Mn, Fe, Zn) from soil sample images.

**After training, download:**
- `composition_model.pth` → place in `models/`
- `training_metrics.json` → place in `models/`

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q torch torchvision pandas openpyxl pillow scikit-learn tqdm matplotlib

In [ ]:
# ── Cell 2: Upload dataset ────────────────────────────────────────────────────
# OPTION A — Mount Google Drive (recommended)
from google.colab import drive
drive.mount('/content/drive')
# After mounting, set SOIL_ROOT to wherever you uploaded the Soil_data_Academia folder
# e.g. '/content/drive/MyDrive/suparco/Soil_data_Academia'

# OPTION B — Upload ZIP manually
# from google.colab import files
# uploaded = files.upload()  # upload 'Soil_data_Academia.zip'
# !unzip Soil_data_Academia.zip -d /content/

In [ ]:
# ── Cell 3: Configure paths ───────────────────────────────────────────────────
# EDIT THIS: path to the Soil_data_Academia folder containing images/ and Soil_Analysis.xlsx
SOIL_ROOT  = '/content/drive/MyDrive/suparco/Soil_data_Academia'  # <-- CHANGE THIS

import os
IMAGE_DIR  = os.path.join(SOIL_ROOT, 'images')
EXCEL_PATH = os.path.join(SOIL_ROOT, 'Soil_Analysis.xlsx')

assert os.path.exists(IMAGE_DIR),  f'images/ not found at {IMAGE_DIR}'
assert os.path.exists(EXCEL_PATH), f'Excel not found at {EXCEL_PATH}'

imgs = [f for f in os.listdir(IMAGE_DIR) if f.endswith('.jpg')]
print(f'Found {len(imgs)} images')

import pandas as pd
df = pd.read_excel(EXCEL_PATH)
print(f'Excel rows: {len(df)}')
print(df.head())

In [ ]:
# ── Cell 4: Imports & Config ──────────────────────────────────────────────────
import json
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision import models
from torchvision.models import ResNet18_Weights
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from tqdm import tqdm
import matplotlib.pyplot as plt

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')
if DEVICE == 'cuda':
    print(torch.cuda.get_device_name(0))

# ── Hyperparameters ────────────────────────────────────────────────────────────
ELEMENTS    = ['Cd', 'Cu', 'Ni', 'Mn', 'Fe', 'Zn']
BATCH_SIZE  = 16
EPOCHS      = 80
LR_HEAD     = 1e-3   # head learning rate
LR_BACKBONE = 1e-4   # backbone learning rate (after unfreeze)
WEIGHT_DECAY = 1e-4
VAL_FRAC    = 0.2
SEED        = 42
UNFREEZE_EPOCH = 10  # epoch at which to unfreeze backbone

torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# ── Cell 5: Dataset ───────────────────────────────────────────────────────────
def get_composition_groups(df):
    """Group sample IDs by identical composition — 19 distinct groups."""
    df = df.copy()
    df['Sample ID'] = df['Sample ID'].astype(str).str.strip()
    df['group'] = df.groupby(ELEMENTS).ngroup()
    return df.groupby('group')['Sample ID'].apply(list).to_dict()

def train_val_split(df, image_dir, val_frac=0.2, seed=42):
    available = {Path(p).stem for p in Path(image_dir).glob('*.jpg')}
    groups = get_composition_groups(df)
    keys = sorted(groups.keys())
    rng = np.random.default_rng(seed)
    rng.shuffle(keys)
    n_val = max(1, round(len(keys) * val_frac))
    val_ids   = [s for k in keys[:n_val]  for s in groups[k] if s in available]
    train_ids = [s for k in keys[n_val:]  for s in groups[k] if s in available]
    return train_ids, val_ids

train_ids, val_ids = train_val_split(df, IMAGE_DIR, val_frac=VAL_FRAC, seed=SEED)
print(f'Train: {len(train_ids)} images | Val: {len(val_ids)} images')

# Label normalisation from training set
train_df = df[df['Sample ID'].isin(train_ids)]
LABEL_MEAN = train_df[ELEMENTS].mean().values.astype(np.float32)
LABEL_STD  = train_df[ELEMENTS].std().values.astype(np.float32)
LABEL_STD  = np.where(LABEL_STD < 1e-6, 1.0, LABEL_STD)
print('Label mean:', LABEL_MEAN)
print('Label std: ', LABEL_STD)

In [ ]:
# ── Cell 6: Dataset class & DataLoaders ──────────────────────────────────────
class SoilDataset(Dataset):
    def __init__(self, df, image_dir, sample_ids, transform, label_mean, label_std):
        self.image_dir   = Path(image_dir)
        self.transform   = transform
        self.label_mean  = label_mean
        self.label_std   = label_std
        available = {p.stem: p for p in self.image_dir.glob('*.jpg')}
        sub = df[df['Sample ID'].astype(str).str.strip().isin(sample_ids)].copy()
        sub['Sample ID'] = sub['Sample ID'].astype(str).str.strip()
        self.records = [
            (available[row['Sample ID']], row[ELEMENTS].values.astype(np.float32))
            for _, row in sub.iterrows()
            if row['Sample ID'] in available
        ]

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        path, labels = self.records[idx]
        img = Image.open(path).convert('RGB')
        img = self.transform(img)
        # normalise labels
        norm_labels = (labels - self.label_mean) / self.label_std
        return img, torch.tensor(norm_labels)

TRAIN_TRANSFORM = T.Compose([
    T.Resize((256, 256)),
    T.RandomCrop(224),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.15, hue=0.05),
    T.RandomRotation(15),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
VAL_TRANSFORM = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

train_ds = SoilDataset(df, IMAGE_DIR, train_ids, TRAIN_TRANSFORM, LABEL_MEAN, LABEL_STD)
val_ds   = SoilDataset(df, IMAGE_DIR, val_ids,   VAL_TRANSFORM,   LABEL_MEAN, LABEL_STD)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
# ── Cell 7: Model ─────────────────────────────────────────────────────────────
class CompositionModel(nn.Module):
    def __init__(self, num_outputs=6, freeze_backbone=True):
        super().__init__()
        backbone = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        in_features = backbone.fc.in_features
        self.features = nn.Sequential(*list(backbone.children())[:-1])
        if freeze_backbone:
            for p in self.features.parameters():
                p.requires_grad = False
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, num_outputs),
        )
    def unfreeze(self):
        for p in self.features.parameters():
            p.requires_grad = True
    def forward(self, x):
        return self.head(self.features(x))

model = CompositionModel(num_outputs=len(ELEMENTS), freeze_backbone=True).to(DEVICE)
criterion = nn.HuberLoss(delta=1.0)  # robust to outliers

# Only train the head initially
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,} | Trainable: {trainable:,}')

In [ ]:
# ── Cell 8: Evaluation helper ─────────────────────────────────────────────────
def evaluate(model, loader, label_mean, label_std, device):
    model.eval()
    all_preds, all_targets = [], []
    total_loss = 0.0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs   = imgs.to(device)
            labels = labels.to(device)
            preds  = model(imgs)
            total_loss += criterion(preds, labels).item() * imgs.size(0)
            # de-normalise
            mean_t = torch.tensor(label_mean, device=device)
            std_t  = torch.tensor(label_std,  device=device)
            all_preds.append((preds * std_t + mean_t).cpu().numpy())
            all_targets.append((labels * std_t + mean_t).cpu().numpy())

    preds_np   = np.vstack(all_preds)    # (N, 6)
    targets_np = np.vstack(all_targets)  # (N, 6)
    preds_np   = np.clip(preds_np, 0, None)

    per_element = {}
    for i, e in enumerate(ELEMENTS):
        per_element[e] = {
            'mae':  float(mean_absolute_error(targets_np[:, i], preds_np[:, i])),
            'rmse': float(np.sqrt(mean_squared_error(targets_np[:, i], preds_np[:, i]))),
            'r2':   float(r2_score(targets_np[:, i], preds_np[:, i])),
        }

    return total_loss / len(loader.dataset), per_element, preds_np, targets_np

In [ ]:
# ── Cell 9: Training loop ─────────────────────────────────────────────────────
train_losses, val_losses = [], []
best_val_loss = float('inf')
best_epoch    = 0

for epoch in range(1, EPOCHS + 1):
    # Unfreeze backbone after UNFREEZE_EPOCH
    if epoch == UNFREEZE_EPOCH:
        model.unfreeze()
        optimizer = torch.optim.AdamW([
            {'params': model.features.parameters(), 'lr': LR_BACKBONE},
            {'params': model.head.parameters(),     'lr': LR_HEAD},
        ], weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS - UNFREEZE_EPOCH
        )
        print(f'Epoch {epoch}: Backbone unfrozen')

    # ── Train ──
    model.train()
    epoch_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        preds = model(imgs)
        loss  = criterion(preds, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item() * imgs.size(0)

    scheduler.step()
    t_loss = epoch_loss / len(train_ds)

    # ── Validate ──
    v_loss, per_elem, _, _ = evaluate(model, val_loader, LABEL_MEAN, LABEL_STD, DEVICE)

    train_losses.append(t_loss)
    val_losses.append(v_loss)

    # Save best
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        best_epoch    = epoch
        torch.save({
            'model_state_dict': model.state_dict(),
            'epoch': epoch,
            'val_loss': v_loss,
            'label_mean': LABEL_MEAN.tolist(),
            'label_std':  LABEL_STD.tolist(),
            'elements': ELEMENTS,
        }, 'composition_model.pth')

    if epoch % 5 == 0 or epoch == 1:
        mean_r2  = np.mean([per_elem[e]['r2']  for e in ELEMENTS])
        mean_mae = np.mean([per_elem[e]['mae'] for e in ELEMENTS])
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Train Loss: {t_loss:.4f} | Val Loss: {v_loss:.4f} | '
              f'Mean R²: {mean_r2:.4f} | Mean MAE: {mean_mae:.4f}')

print(f'\nBest model at epoch {best_epoch}, val_loss={best_val_loss:.4f}')

In [ ]:
# ── Cell 10: Final evaluation & metrics ──────────────────────────────────────
# Reload best checkpoint
ckpt = torch.load('composition_model.pth', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])

val_loss, per_elem, preds_np, targets_np = evaluate(
    model, val_loader, LABEL_MEAN, LABEL_STD, DEVICE
)

print('\n' + '='*60)
print('VALIDATION SET PERFORMANCE (Best Model)')
print('='*60)
print(f'{"Element":<12} {"MAE":>10} {"RMSE":>10} {"R²":>10}')
print('-'*45)
r2_list, mae_list, rmse_list = [], [], []
for e in ELEMENTS:
    m = per_elem[e]
    print(f'{e:<12} {m["mae"]:>10.4f} {m["rmse"]:>10.4f} {m["r2"]:>10.4f}')
    r2_list.append(m['r2'])
    mae_list.append(m['mae'])
    rmse_list.append(m['rmse'])
print('-'*45)
print(f'{"MEAN":<12} {np.mean(mae_list):>10.4f} {np.mean(rmse_list):>10.4f} {np.mean(r2_list):>10.4f}')

# Save metrics JSON
metrics_out = {
    'overall': {
        'mean_r2':   float(np.mean(r2_list)),
        'mean_mae':  float(np.mean(mae_list)),
        'mean_rmse': float(np.mean(rmse_list)),
        'best_epoch': best_epoch,
        'val_loss':   float(best_val_loss),
    },
    'per_element': per_elem,
    'train_losses': [float(x) for x in train_losses],
    'val_losses':   [float(x) for x in val_losses],
    'elements': ELEMENTS,
    'label_mean': LABEL_MEAN.tolist(),
    'label_std':  LABEL_STD.tolist(),
}
with open('training_metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print('\nMetrics saved to training_metrics.json')

In [ ]:
# ── Cell 11: Visualise results ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, e in enumerate(ELEMENTS):
    ax = axes[i]
    t, p = targets_np[:, i], preds_np[:, i]
    ax.scatter(t, p, alpha=0.7, s=50, c='steelblue', edgecolors='white', linewidth=0.5)
    lo, hi = min(t.min(), p.min()) * 0.9, max(t.max(), p.max()) * 1.1
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect prediction')
    m = per_elem[e]
    ax.set_title(
        f'{e} — R²={m["r2"]:.3f} | MAE={m["mae"]:.3f} | RMSE={m["rmse"]:.3f}',
        fontsize=10
    )
    ax.set_xlabel('Ground Truth (mg/kg)')
    ax.set_ylabel('Predicted (mg/kg)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('Predicted vs Ground Truth — Validation Set', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('prediction_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

# Loss curve
fig2, ax2 = plt.subplots(figsize=(10, 4))
ax2.plot(train_losses, label='Train Loss', color='steelblue')
ax2.plot(val_losses,   label='Val Loss',   color='orange', linestyle='--')
ax2.axvline(best_epoch - 1, color='green', linestyle=':', label=f'Best Epoch ({best_epoch})')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Huber Loss')
ax2.set_title('Training & Validation Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 12: Download trained model ──────────────────────────────────────────
from google.colab import files
files.download('composition_model.pth')
files.download('training_metrics.json')
files.download('prediction_scatter.png')
files.download('loss_curve.png')
print('Downloads started — place .pth and .json in the models/ folder')